# CellExLink: reproducible software demonstration

**Cell Ontology-linked cell-type recognition and normalization from biomedical text**

[![Open in Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShahriyariLab/CellExLink/blob/main/examples/demo.ipynb)

This notebook accompanies the CellExLink SoftwareX software article. It demonstrates the user-facing software workflows.

CellExLink can recognize cell-type mentions, link them to Cell Ontology concepts, display the predictions in a notebook, and save structured results.

**Project resources:** [source code](https://github.com/ShahriyariLab/CellExLink) · [documentation](https://shahriyarilab.github.io/CellExLink/) · GPL-3.0 license


## Notebook overview

The `CellExLinkPipeline` class provides four high-level methods for common forms of biomedical text input.

| Python method | Typical input | Main output |
|---|---|---|
| `run_text` | One text string | Python result objects; optional compact JSON |
| `run_bioc` | A BioC XML/JSON document or collection | BioC XML or BioC JSON |
| `run_files` | Local files, directories, or both | One result file for each input file |
| `run_pmids` | One or more PubMed or PubMed Central identifiers | One merged BioC result collection |

The examples below cover:

1. end-to-end and recognition-only processing of plain text;
2. retrieval and processing of a publication list;
3. batch processing of separate local files;
4. preservation of PubTator3 annotations while adding CellExLink cell types;
5. notebook visualization and saved outputs.

> **Task values:** use `task="ner"` for recognition only, `task="nen"` for normalization of existing spans in structured input, and `task="end-to-end"` for recognition followed by normalization. Raw text does not contain existing spans, so `run_text` supports only `"ner"` and `"end-to-end"`.


## Before you begin

- Run the notebook from top to bottom in a fresh Python or Colab session.
- Internet access is required to install the package, download the model checkpoints.
- A GPU is recommended for faster inference, but the examples can run on a CPU.
- The first end-to-end prediction takes longer because CellExLink loads the recognition and normalization components and encodes the ontology resources. Later calls on the same pipeline object reuse those resources automatically.
- The notebook writes all generated files under `cellexlink_demo/`, so the working directory remains organized.



## 1. Install CellExLink

### Install CellExLink, check the runtime, and create working directories

The following cell installs CellExLink, creates separate folders for models, inputs, and outputs files. This makes the notebook easier to
reproduce and keeps generated files out of the source directory.


In [ ]:
# Install CellExLink from PyPI.
!python -m pip install cellexlink

In [ ]:
import json
import platform
from importlib.metadata import version
from pathlib import Path

import torch

# Use /content in Colab and the current directory in other notebook environments.
BASE_DIR = Path("/content") if Path("/content").exists() else Path.cwd()
WORK_DIR = BASE_DIR
MODEL_DIR = WORK_DIR / "models"
INPUT_DIR = WORK_DIR / "inputs"
OUTPUT_DIR = WORK_DIR / "outputs"

for directory in (MODEL_DIR, INPUT_DIR, OUTPUT_DIR, RUNTIME_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print("Python:", platform.python_version())
print("CellExLink:", version("cellexlink"))
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Working directory:", WORK_DIR)


## 2. Download the released model checkpoints

The CellExLink package does not include the neural network model weights. The command below downloads the default named entity recognition checkpoint, `CellExLink-bioformer16L`, and the named entity normalization checkpoint, `CellExLink-Sapbert`, into the `models` directory.


In [ ]:
!cellexlink download-models --output-dir "{MODEL_DIR}"

## 3. Create one reusable pipeline

Create the pipeline once and reuse it throughout the notebook. CellExLink initializes only the components required by the first task and retains the loaded models, ontology embeddings, and static abbreviation resources for later calls. Document-specific abbreviation mappings are rebuilt for each document or processing group.


In [ ]:
from IPython.display import display
from cellexlink import (
    CellExLinkPipeline,
    render_cell_type_annotations,
    write_predictions_json,
)

pipe = CellExLinkPipeline.from_pretrained()

print("Pipeline created. Models will load automatically on first use.")


## 4. Example 1 — Plain text, visual inspection, and saved JSON

This example uses a biomedical passage containing full cell-type names. `run_text` returns Python objects with the detected mention, character span, and—during end-to-end processing—the predicted Cell Ontology identifier and label.

The notebook display preserves the original text and highlights each detected mention. Move the pointer over a highlighted span to view its Cell Ontology label and identifier.

In [ ]:
text = (
    "In colon cancer, activated CD8+ T cells enhance production of necrotic cells "
    "by expressing high levels of cytokines like IFN-γ and FasL. Necrotic cells "
    "and macrophages release HMGB1 to activate dendritic cells, which leads to "
    "activation of T cells. In addition, intestinal epithelial cells, which are "
    "in close contact with DCs, activate dendritic cells by releasing molecules "
    "like thymic stromal lymphopoietin (TSLP). Once activated, dendritic cells "
    "release cytokines STAT4, STAT6, and IL-4, which induce differentiation of "
    "naive T cells into effector T cells (Th1, Th17, and Th2). CD4+ T cells can "
    "also become activated by TNF-α, which is released by M1 macrophages. Activated "
    "CD4+ T cells release IL-2, IL-4, IL-5, IL-13, and IL-17 to activate killer "
    "cells such as CD8+ T cells. CD4+ T cells also release IFN-γ, which activates "
    "M1 macrophages. Activated macrophages and CD4+ effector T cells release the "
    "tumor-promoting cytokine interleukin 6 (IL-6). IL-6 promotes tumor growth by "
    "activating STAT3 in intestinal epithelial cells."
)


In [ ]:
# Detect and normalize cell-type mentions in the text.
results = pipe.run_text(
    text,
    task="end-to-end",  # Use "ner" for recognition only.
)

# Display highlighted mentions and their Cell Ontology links.
display(render_cell_type_annotations(text, results))

# Save the results as compact JSON.
plain_text_json = OUTPUT_DIR / "cell_types.json"
write_predictions_json(
    results,
    plain_text_json,
)

print(f"Saved: {plain_text_json}")


### Recognition-only mode

Recognition-only mode returns mention spans without Cell Ontology links. The same NER component loaded above is reused.


In [ ]:
ner_results = pipe.run_text(text, task="ner")

print(
    json.dumps(
        [item.to_dict() for item in ner_results],
        indent=2, ensure_ascii=False,
    )
)


### Command-line equivalent

The command-line interface uses the same processing implementation. A shell-based plain-text run can be written as:

```bash
cellexlink predict-text \
  --text "CD8+ T cells infiltrated the tumor." \
  --task end-to-end \
  --output outputs/cell_types.json
```

## 5. Example 2 — Process a list of PubMed identifiers

Many literature projects begin with PubMed or PubMed Central identifiers rather than downloaded article files. `run_pmids` retrieves available records from NCBI or Europe PMC, applies the selected task, and writes one merged BioC collection. The repository includes an example publication identifier list at examples/PMID_list.txt. Copy this file to INPUT_DIR before running the following example.

The public call remains simple for small and large lists. Two optional arguments control internal processing:

- `chunk_size`: maximum number of publication identifiers retrieved in one group;
- `bioc_chunk_size`: maximum number of BioC passages processed in one inference chunk.

The loaded models and static ontology resources remain available across all groups. The example below reads identifiers from a text file, matching a common reproducible literature workflow.


In [ ]:
pmid_file = INPUT_DIR / "PMID_list.txt"

ids = pmid_file.read_text(encoding="utf-8").split()

pmid_output = OUTPUT_DIR / "pmid_results.bioc.xml"
pipe.run_pmids(
    ids,
    pmid_output,
    task="end-to-end",
    text_source="abstract",  # Use "fulltext" when available from the source.
    chunk_size=100,           # Maximum identifiers per retrieval group.
    bioc_chunk_size=256,      # Maximum passages per inference chunk.
)

print(f"Requested identifiers: {len(ids)}")
print(f"Saved merged collection: {pmid_output}")


In [ ]:
pmid_results = pipe.read_predictions_from_bioc(pmid_output)

print(
    json.dumps(
        [item.to_dict() for item in pmid_results[:5]],
        indent=2,
        ensure_ascii=False,
    )
)

# The package can render CellExLink annotations directly from BioC output.
display(render_cell_type_annotations(pmid_output))


## 6. Example 3

 — Add CellExLink annotations to PubTator3 output

PubTator3 is a widely used biomedical literature resource that supplies article text and annotations for several entity classes. However, it does not provide a dedicated entity class for biological cell types or link cell-type mentions to Cell Ontology. CellExLink can therefore complement PubTator3 by preserving its existing annotations and adding specialized cell-type mentions and Cell Ontology identifiers to the same BioC document.

This example downloads the PubTator3 BioC JSON export for PMID 30243656. A request made with the `pmids` parameter normally returns the title and abstract. Full-text PubTator3 exports require an available PubMed Central record and the `pmcids` parameter.

In [ ]:
from urllib.request import Request, urlopen

pubtator_input = INPUT_DIR / "pubtator3_annotations.bioc.json"

!curl -L "https://www.ncbi.nlm.nih.gov/research/pubtator3-api/publications/export/biocjson?pmids=30243656" \
  -o "{pubtator_input}"

In [ ]:
combined_output = OUTPUT_DIR / "pubtator3_plus_cellexlink.bioc.json"

pipe.run_bioc(
    pubtator_input,
    combined_output,
    task="end-to-end",
    input_format="bioc-json",
    output_format="bioc-json",
    preserve_existing_annotations=True,
    chunk_size=256,  # Maximum passages per inference chunk.
)

print("Saved combined BioC document:", pubtator_output)


### Inspect the complementary annotation layers

The package includes a notebook renderer for merged PubTator3 and CellExLink BioC JSON. PubTator3 entities appear in yellow, and CellExLink cell-type annotations appear in purple. Hover over a highlight to view the source, entity type, label, and identifier.


In [ ]:
from cellexlink.mics.pubtator3_cellExlink_display import (
    render_cellexlink_bioc_file,
)

display(render_cellexlink_bioc_file(combined_output))

## 7. Command-line workflows

The package exposes corresponding command-line operations for shell scripts and batch systems.

```bash
# Plain text
cellexlink predict-text --text "CD8+ T cells infiltrated the tumor." \
  --task end-to-end --output text_results.json

# Structured BioC input
cellexlink run-bioc input.bioc.json output.bioc.json \
  --task end-to-end --preserve-existing-annotations

# Multiple local files, with one output per input
cellexlink run-files corpus/ --results-dir results/ \
  --task end-to-end --chunk-size 100 --bioc-chunk-size 256

# Publication identifiers, merged into one BioC output
cellexlink predict-pmid --ids-file PMID_list.txt \
  --output pmid_results.bioc.xml --text-source abstract
```

Run the next cell to inspect all currently available commands and options.


In [ ]:
subprocess.run(["cellexlink", "--help"], check=True)